# 01 — Reprodução: detecção de mixers na Ethereum

Notebook didático de reprodução do projeto inspirado no artigo *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (SBSeg 2025).

**Escopo desta versão:** utilizar datasets processados e auditados, verificar integridade, validar e treinar Random Forest, e avaliar o dia 29/10/2020. **Não reprocessa dumps brutos** e não deve ser apresentado como validação cega. Código versionado no [GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/tree/main/src/ethmix); esta cópia é para execução no Colab.

**Unidade:** janela contrato-destinatário/dia UTC. Um endereço pode produzir várias janelas. Conjunto de treino legado tem 23 positivas de **um único pool** Tornado Cash 10 ETH em março/2025 e 100 negativas selecionadas por inspeção; isso limita a generalização.
**Dados oficiais:** Projeto_Mixers_Ethereum no seu Drive (não o atalho do Bruno).


## 1 — Conceitos e variáveis

| Feature | O que mede |
|---|---|
| `value_entropy` | Diversidade dos valores transferidos |
| `fee_entropy` | Diversidade das taxas |
| `internal_value_entropy` | Diversidade dos valores internos |
| `call_count_mean` | Média das chamadas da transação |

As entropias são Shannon base 2, depois de discretização por arredondamento (4 casas valores, 6 taxas). Parâmetros de engenharia deste projeto não são automaticamente parâmetros explícitos do artigo.

## 2 — Montar o Drive e carregar o código da pesquisa

Os módulos `ethmix` estão na **sua pasta oficial do Google Drive**, em `Projeto_Mixers_Ethereum/src/ethmix/`, sincronizados com `src/ethmix/` do GitHub. **Não precisa criar token nem acessar o GitHub para executar**.

A célula monta o Drive, localiza a sua pasta e **compara SHA-256 dos três módulos com a versão homologada no GitHub**. Se alguém alterar o código no Drive sem sincronizá-lo, a execução será interrompida para evitar resultados não reproduzíveis.

> O GitHub continua sendo a fonte oficial de versionamento; o Drive hospeda a cópia operacional de leitura para o Colab. Não edite os dois separadamente.


In [ ]:
from pathlib import Path
from google.colab import drive
import json, sys, hashlib

drive.mount('/content/drive')
REPO = 'BorgesGusta/ethereum-mixer-detection'
CODIGO_REFERENCIA = 'src/ethmix (GitHub, versão conferida por SHA-256)'
BASE_MANUAL = None  # apenas se o Drive estiver montado em caminho diferente
BASE_OFICIAL = Path('/content/drive/MyDrive/Segurança de Redes de Computadores/Projeto_Mixers_Ethereum')
BASE = Path(BASE_MANUAL) if BASE_MANUAL else BASE_OFICIAL
if not BASE.is_dir():
    raise FileNotFoundError(f'Pasta oficial não encontrada: {BASE}. Defina BASE_MANUAL, se necessário.')

EXPECTED_MODULE_SHA256 = {'__init__.py': 'dccb63bbe5fbde892d3059dab7e3efc03f2e0e169d93afa67e8d983e996cd1ae', 'features.py': 'b75d65bd2326169e6fc52f86f5030337d6e6813df607bab0a9c3cb20922fd0b5', 'models.py': '7f92d88cafcfc7c939f8c84e4fcfb34a0aa98c0890a86333c294b23f1508d6d9'}
for filename, esperado in EXPECTED_MODULE_SHA256.items():
    arquivo = BASE / 'src' / 'ethmix' / filename
    if not arquivo.is_file():
        raise FileNotFoundError(f'Módulo ausente no seu Drive: {arquivo}')
    obtido = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    if obtido != esperado:
        raise ValueError(f'Código diferente do GitHub: {filename}. Não execute até sincronizar.')

sys.path.insert(0, str(BASE / 'src'))
from ethmix.features import FEATURES, DECIMAIS, preparar, calcular_features
from ethmix.models import novo_rf, validar_cv, reamostrar
print('Módulos carregados do seu Google Drive:', list(FEATURES))
print('Pasta oficial:', BASE)
print('Integridade dos três módulos: verificada.')


## 3 — Ler os dados processados e conferir integridade

Usaremos **somente os arquivos copiados para seu Drive oficial**. Os datasets pesados continuam armazenados em `data/processed/` e são verificados com SHA-256 contra as referências previamente registradas. O notebook interrompe a execução se detectar diferenças.

Nenhum dado de treinamento original é sobrescrito.


In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timezone

INPUTS = {
    'treino': BASE / 'data/processed/features_treino.parquet',
    'teste2020': BASE / 'data/processed/features_teste_20201029.parquet',
}
HASHES_ESPERADOS = {
    'treino': '0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32',
    'teste2020': '6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51',
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(4*1024*1024), b''):
            h.update(bloco)
    return h.hexdigest()

for chave, arquivo in INPUTS.items():
    if not arquivo.is_file():
        raise FileNotFoundError(f'Arquivo ausente na pasta oficial: {arquivo}')
    obtido = sha256(arquivo)
    print(chave, arquivo.name, obtido)
    if obtido != HASHES_ESPERADOS[chave]:
        raise ValueError(f'SHA-256 inesperado para {chave}. Verifique a cópia antes de continuar.')
print('Datasets processados: SHA-256 conferido.')


## 4 — Inspecionar as amostras

**Treino (legado):** 123 janelas, 23 positivas e 100 negativas, com positivos de um único contrato. A validação cruzada estratificada por linha não comprova generalização entre contratos.

**Teste (29/10/2020):** apenas contratos com pelo menos 10 transações entram na classificação. Os demais endereços não são negativos verificados.

In [ ]:
treino=pd.read_parquet(INPUTS["treino"])
teste=pd.read_parquet(INPUTS["teste2020"])
MIN_TX=10
SEED=42
assert set(FEATURES).issubset(treino.columns) and set(FEATURES).issubset(teste.columns)
assert {"label","n_tx"}.issubset(treino.columns)
assert {"recipient","n_tx","tornado"}.issubset(teste.columns)
assert set(treino.label.dropna().unique())=={0,1}
treino_e=treino.loc[treino.n_tx>=MIN_TX].copy()
teste_e=teste.loc[teste.n_tx>=MIN_TX].copy()
print("Treino completo:",len(treino),"classes:",treino.label.value_counts().to_dict())
print("Treino elegível:",len(treino_e),"classes:",treino_e.label.value_counts().to_dict())
print("Teste 2020:",len(teste),"janelas; elegíveis:",len(teste_e))
print("Pools de referência:",int(teste_e.tornado.notna().sum()))
display(treino_e.groupby("label")[list(FEATURES)].median().round(3))

## 5 — Validar e treinar o classificador

Random Forest com 100 árvores, profundidade máxima 5, pesos balanceados e semente 42. Oversampling apenas dentro dos folds de treino. As métricas são **internas e exploratórias**, não evidência de teste independente.

In [ ]:
cv=validar_cv(treino_e,seed=SEED,n_splits=5)
display(cv.round(3))
display(cv[["precision","recall","f1"]].agg(["mean","std"]).round(3))
Xt,yt=reamostrar(treino_e[list(FEATURES)],treino_e.label.astype(int),SEED)
modelo=novo_rf(SEED).fit(Xt,yt)
print("RF treinada",modelo.n_estimators,"árvores; max_depth",modelo.max_depth)

## 6 — Aplicar ao dia histórico de 2020

Testamos limiares 0,50 e 0,85 **apenas para caracterização exploratória**. Os demais alertas são endereços sem rótulos confirmados; não podem ser chamados automaticamente de falsos positivos.

A referência de 63 endereços no artigo é um ponto de comparação, **não** um alvo de otimização.

In [ ]:
prob=modelo.predict_proba(teste_e[list(FEATURES)])[:,1]
rows=[]
for limiar in [0.50,0.85]:
    mask=prob>limiar
    recuperados=int(teste_e.loc[mask,"tornado"].notna().sum())
    rows.append({"limiar":limiar,"enderecos_sinalizados":int(mask.sum()),"pools_referencia_recuperados":recuperados,"outros_sinalizados_sem_rotulo_confirmado":int(mask.sum())-recuperados})
resumo=pd.DataFrame(rows)
display(resumo)
print("Confrontar estes números com os relatórios originais antes de afirmar paridade")

## 7 — Registrar a execução sem sobrescrever o legado

Cada execução cria uma nova pasta em `results/reproducao_consolidada/`. Para validar a consolidação, comparar os CSVs recém-produzidos com as métricas e alertas dos experimentos legados. Não salvar sobre o modelo antigo.

In [ ]:
ts=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=BASE/'results'/'reproducao_consolidada'/ts
OUT.mkdir(parents=True,exist_ok=False)
cv.to_csv(OUT/'cv_folds.csv',index=False)
resumo.to_csv(OUT/'resumo_2020.csv',index=False)
manifesto={
    'experimento':'reproducao_consolidada_01',
    'data_utc':ts,
    'repo':REPO,
    'codigo_ref':CODIGO_REFERENCIA,
    'codigo_sha256':EXPECTED_MODULE_SHA256,
    'dados_sha256':HASHES_ESPERADOS,
    'min_tx':MIN_TX,'seed':SEED,
    'rf':{'n_estimators':100,'max_depth':5,'class_weight':'balanced'},
    'limiares_exploratorios':[0.50,0.85],
    'ressalvas':['positivos de um unico contrato',
                 'enderecos nao rotulados nao sao negativos',
                 'teste 2020 previamente explorado']
}
(OUT/'manifesto.json').write_text(json.dumps(manifesto,indent=2,ensure_ascii=False),encoding='utf-8')
print('Resultados:',OUT)
